# Day 2 — SLTE Framework Pipeline
**Paper:** Can Synthetic Data Replace Real Data? (SLTE Framework)

### What this notebook does
- **Stage 1:** Train anchor model on real data → compute Label Confidence Score (LCS) for every synthetic sample
- **Stage 2:** kNN label correction for low-confidence synthetic samples
- **Stage 3:** Uncertainty-stratified routing (HIGH / MID / LOW confidence tracks)
- **TSTR re-run:** 5 classifiers × 6 mixing ratios using SLTE-corrected synthetic data
- **Delta computation:** SLTE vs No-SLTE performance gap

### Files needed (upload to Colab before running)
- `MedicalAppointment.csv` (original dataset)
- `synthetic_full.csv` (from Day 1)
- `tstr_results.csv` (from Day 1)
- `baseline_results.json` (from Day 1)
---

In [ ]:
# CELL 1 — Install & Imports
!pip install -q scikit-learn pandas numpy matplotlib openpyxl

import pandas as pd
import numpy as np
import json, warnings
from datetime import datetime

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              precision_score, recall_score)
from sklearn.base import clone
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Imports done')

✅ Imports done


In [ ]:
# ════════════════════════════════════════════════
# CELL 2 — Load all Day 1 files
# ════════════════════════════════════════════════
# Load original dataset and re-preprocess to get train/test split
df_raw = pd.read_csv('MedicalAppointment.csv')
df = df_raw.copy()
df.drop(columns=['PatientId', 'AppointmentID'], inplace=True)
df = df[df['Age'] >= 0].copy()
df['ScheduledDay']   = pd.to_datetime(df['ScheduledDay'],   utc=True)
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'], utc=True)
df['WaitDays']               = (df['AppointmentDay'] - df['ScheduledDay']).dt.days.clip(lower=0)
df['ScheduledDayOfWeek']     = df['ScheduledDay'].dt.dayofweek
df['AppointmentDayOfWeek']   = df['AppointmentDay'].dt.dayofweek
df['ScheduledMonth']         = df['ScheduledDay'].dt.month
df.drop(columns=['ScheduledDay', 'AppointmentDay'], inplace=True)
df['Gender'] = (df['Gender'] == 'M').astype(int)
freq_map = df['Neighbourhood'].value_counts(normalize=True).to_dict()
df['Neighbourhood_freq'] = df['Neighbourhood'].map(freq_map)
df.drop(columns=['Neighbourhood'], inplace=True)
df['No-show'] = (df['No-show'] == 'Yes').astype(int)

TARGET   = 'No-show'
FEATURES = [c for c in df.columns if c != TARGET]
X = df[FEATURES]
y = df[TARGET]

X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

# Load Day 1 synthetic data
synth_df       = pd.read_csv('synthetic_full.csv')
X_synth        = synth_df[FEATURES]
y_synth        = synth_df[TARGET]

# Load Day 1 TSTR results (no SLTE)
df_tstr_no_slte = pd.read_csv('tstr_results.csv')
with open('baseline_results.json') as f:
    baseline_results = json.load(f)

print(f'Real train: {X_train_real.shape[0]:,}  |  Real test: {X_test_real.shape[0]:,}')
print(f'Synthetic : {X_synth.shape[0]:,}')
print(f'Features  : {FEATURES}')
print(f'\nSynthetic target dist: {y_synth.value_counts().to_dict()}')
print(f'Real train target dist: {y_train_real.value_counts().to_dict()}')
print('\n✅ All Day 1 files loaded')

Real train: 88,420  |  Real test: 22,106
Synthetic : 88,420
Features  : ['Gender', 'Age', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'WaitDays', 'ScheduledDayOfWeek', 'AppointmentDayOfWeek', 'ScheduledMonth', 'Neighbourhood_freq']

Synthetic target dist: {0: 74122, 1: 14298}
Real train target dist: {0: 70565, 1: 17855}

✅ All Day 1 files loaded


In [ ]:
# ════════════════════════════════════════════════
# CELL 3 — SLTE STAGE 1: Label Confidence Score
# ════════════════════════════════════════════════
print('SLTE Stage 1: Training anchor model on real data...')
print('  Using RandomForest (150 trees) as anchor — best AUC on real data')

anchor_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
anchor_model.fit(X_train_real, y_train_real)

# Predict probability of each synthetic sample's label being correct
synth_proba = anchor_model.predict_proba(X_synth)   # shape (n, 2)

# LCS = probability anchor assigns to the ACTUAL synthetic label
lcs = np.array([
    synth_proba[i, int(y_synth.iloc[i])]
    for i in range(len(y_synth))
])

print(f'\n  LCS Statistics:')
print(f'    Mean  : {lcs.mean():.4f}')
print(f'    Std   : {lcs.std():.4f}')
print(f'    Min   : {lcs.min():.4f}')
print(f'    Max   : {lcs.max():.4f}')
print(f'    Median: {np.median(lcs):.4f}')

LOW_CONF_THRESH  = 0.40   # below this → label correction applied
HIGH_CONF_THRESH = 0.70   # above this → fully trusted

low_mask  = lcs <  LOW_CONF_THRESH
high_mask = lcs >= HIGH_CONF_THRESH
mid_mask  = ~low_mask & ~high_mask

print(f'\n  Confidence track assignment:')
print(f'    HIGH (LCS ≥ 0.70): {high_mask.sum():,} samples ({high_mask.mean()*100:.1f}%)')
print(f'    MID  (0.40–0.70) : {mid_mask.sum():,}  samples ({mid_mask.mean()*100:.1f}%)')
print(f'    LOW  (LCS < 0.40): {low_mask.sum():,}  samples ({low_mask.mean()*100:.1f}%)')
print('\n✅ Stage 1 complete — LCS computed for all synthetic samples')

SLTE Stage 1: Training anchor model on real data...
  Using RandomForest (150 trees) as anchor — best AUC on real data

  LCS Statistics:
    Mean  : 0.6977
    Std   : 0.2392
    Min   : 0.0000
    Max   : 1.0000
    Median: 0.7533

  Confidence track assignment:
    HIGH (LCS ≥ 0.70): 53,139 samples (60.1%)
    MID  (0.40–0.70) : 22,666  samples (25.6%)
    LOW  (LCS < 0.40): 12,615  samples (14.3%)

✅ Stage 1 complete — LCS computed for all synthetic samples


In [ ]:
# ════════════════════════════════════════════════
# CELL 4 — SLTE STAGE 2: kNN Label Correction
# ════════════════════════════════════════════════
print('SLTE Stage 2: Topology-aware kNN label correction...')
print(f'  Correcting {low_mask.sum():,} low-confidence samples (LCS < {LOW_CONF_THRESH})')

K = 7   # neighbourhood size — odd number avoids ties

# Fit kNN on real training data (topology anchor)
knn = NearestNeighbors(n_neighbors=K, metric='euclidean', n_jobs=-1)
knn.fit(X_train_real)

# Correct low-confidence synthetic labels
y_synth_corrected = y_synth.copy().reset_index(drop=True)
X_synth_reset     = X_synth.reset_index(drop=True)

X_low = X_synth_reset[low_mask]
_, neighbour_indices = knn.kneighbors(X_low)

n_corrected = 0
for i, idx_set in zip(X_low.index, neighbour_indices):
    neighbour_labels = y_train_real.iloc[idx_set].values
    majority_label   = int(np.round(neighbour_labels.mean()))
    if y_synth_corrected.iloc[i] != majority_label:
        y_synth_corrected.iloc[i] = majority_label
        n_corrected += 1

print(f'\n  Labels corrected : {n_corrected:,}')
print(f'  Labels unchanged : {low_mask.sum() - n_corrected:,}')
print(f'  Correction rate  : {n_corrected/low_mask.sum()*100:.1f}% of low-conf samples')
print(f'  As % of all synth: {n_corrected/len(y_synth)*100:.1f}%')
print(f'\n  Original label dist : {pd.Series(y_synth.values).value_counts().to_dict()}')
print(f'  Corrected label dist: {y_synth_corrected.value_counts().to_dict()}')
print('\n✅ Stage 2 complete — low-confidence labels corrected via kNN')

SLTE Stage 2: Topology-aware kNN label correction...
  Correcting 12,615 low-confidence samples (LCS < 0.4)

  Labels corrected : 10,794
  Labels unchanged : 1,821
  Correction rate  : 85.6% of low-conf samples
  As % of all synth: 12.2%

  Original label dist : {0: 74122, 1: 14298}
  Corrected label dist: {0: 84176, 1: 4244}

✅ Stage 2 complete — low-confidence labels corrected via kNN


In [ ]:
# ════════════════════════════════════════════════
# CELL 5 — SLTE STAGE 3: Uncertainty-Stratified Routing
# ════════════════════════════════════════════════
print('SLTE Stage 3: Assigning sample weights by confidence track...')

# Sample weight scheme:
#   HIGH confidence → fully trusted   (w = 1.0)
#   MID  confidence → semi-trusted    (w = 0.7)
#   LOW  confidence → corrected/noisy (w = 0.5)
sample_weights_synth = np.ones(len(y_synth))
sample_weights_synth[mid_mask]  = 0.7
sample_weights_synth[low_mask]  = 0.5

print(f'  HIGH track (w=1.0): {high_mask.sum():,} samples')
print(f'  MID  track (w=0.7): {mid_mask.sum():,}  samples')
print(f'  LOW  track (w=0.5): {low_mask.sum():,}  samples (labels corrected in Stage 2)')
print(f'  Effective weighted size: {sample_weights_synth.sum():,.0f}')

# Save LCS data for Day 3 analysis and figures
lcs_df = pd.DataFrame({
    'lcs'             : lcs,
    'original_label'  : y_synth.values,
    'corrected_label' : y_synth_corrected.values,
    'track'           : np.where(high_mask,'HIGH', np.where(mid_mask,'MID','LOW')),
    'sample_weight'   : sample_weights_synth,
    'label_changed'   : (y_synth.values != y_synth_corrected.values).astype(int)
})
lcs_df.to_csv('lcs_distribution.csv', index=False)

print('\n✅ Stage 3 complete — lcs_distribution.csv saved')

SLTE Stage 3: Assigning sample weights by confidence track...
  HIGH track (w=1.0): 53,139 samples
  MID  track (w=0.7): 22,666  samples
  LOW  track (w=0.5): 12,615  samples (labels corrected in Stage 2)
  Effective weighted size: 75,313

✅ Stage 3 complete — lcs_distribution.csv saved


In [ ]:
# ════════════════════════════════════════════════
# CELL 6 — TSTR WITH SLTE (5 classifiers × 6 ratios)
# ════════════════════════════════════════════════
CLASSIFIERS = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Naive Bayes'         : GaussianNB(),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=10, random_state=42, class_weight='balanced'),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
}
MIXING_RATIOS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
n_train = len(X_train_real)
all_slte = []

def evaluate(clf, X_tr, y_tr, X_te, y_te, weights=None):
    try:
        clf.fit(X_tr, y_tr, sample_weight=weights)
    except TypeError:
        clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:,1] if hasattr(clf,'predict_proba') else y_pred.astype(float)
    return {
        'accuracy' : round(accuracy_score(y_te, y_pred), 4),
        'f1'       : round(f1_score(y_te, y_pred),       4),
        'precision': round(precision_score(y_te, y_pred, zero_division=0), 4),
        'recall'   : round(recall_score(y_te, y_pred),   4),
        'auc_roc'  : round(roc_auc_score(y_te, y_prob),  4),
    }

print(f'Running TSTR with SLTE: {len(CLASSIFIERS)} classifiers × {len(MIXING_RATIOS)} ratios...\n')

for ratio in MIXING_RATIOS:
    n_s = int(n_train * ratio)
    n_r = n_train - n_s

    if n_r > 0 and n_s > 0:
        Xr = X_train_real.sample(n=n_r, random_state=42)
        yr = y_train_real.loc[Xr.index]
        wr = np.ones(n_r)
        s_idx = np.random.choice(len(X_synth_reset), size=n_s, replace=False)
        Xs = X_synth_reset.iloc[s_idx]
        ys = y_synth_corrected.iloc[s_idx]
        ws = sample_weights_synth[s_idx]
        Xm = pd.concat([Xr.reset_index(drop=True), Xs.reset_index(drop=True)]).reset_index(drop=True)
        ym = pd.concat([yr.reset_index(drop=True), ys.reset_index(drop=True)]).reset_index(drop=True)
        wm = np.concatenate([wr, ws])
    elif n_s == 0:
        Xm = X_train_real.reset_index(drop=True)
        ym = y_train_real.reset_index(drop=True)
        wm = np.ones(n_train)
    else:
        s_idx = np.random.choice(len(X_synth_reset), size=n_train, replace=False)
        Xm = X_synth_reset.iloc[s_idx].reset_index(drop=True)
        ym = y_synth_corrected.iloc[s_idx].reset_index(drop=True)
        wm = sample_weights_synth[s_idx]

    print(f'ratio={ratio:.1f}  (real={n_r:,} | synth={n_s:,})')
    for name, clf in CLASSIFIERS.items():
        res = evaluate(clone(clf), Xm, ym, X_test_real, y_test_real, wm)
        all_slte.append({'synthetic_ratio':ratio,'n_real':n_r,'n_synthetic':n_s,
                         'classifier':name,'slte':True,**res})
        print(f'    {name:25s}  Acc={res["accuracy"]:.4f}  F1={res["f1"]:.4f}  AUC={res["auc_roc"]:.4f}')
    print()

df_slte = pd.DataFrame(all_slte)
df_slte.to_csv('slte_results.csv', index=False)
print('✅ SLTE TSTR complete — slte_results.csv saved')

Running TSTR with SLTE: 5 classifiers × 6 ratios...

ratio=0.0  (real=88,420 | synth=0)
    Logistic Regression        Acc=0.6613  F1=0.4105  AUC=0.6659
    Naive Bayes                Acc=0.7665  F1=0.2294  AUC=0.6496
    Decision Tree              Acc=0.6054  F1=0.4379  AUC=0.7154
    Random Forest              Acc=0.7751  F1=0.2539  AUC=0.7135
    Gradient Boosting          Acc=0.7980  F1=0.0062  AUC=0.7298

ratio=0.2  (real=70,736 | synth=17,684)
    Logistic Regression        Acc=0.6567  F1=0.4102  AUC=0.6582
    Naive Bayes                Acc=0.7450  F1=0.3008  AUC=0.6385
    Decision Tree              Acc=0.5744  F1=0.4353  AUC=0.7117
    Random Forest              Acc=0.7787  F1=0.2245  AUC=0.7087
    Gradient Boosting          Acc=0.7981  F1=0.0000  AUC=0.7229

ratio=0.4  (real=53,052 | synth=35,368)
    Logistic Regression        Acc=0.6205  F1=0.3926  AUC=0.6479
    Naive Bayes                Acc=0.7284  F1=0.3330  AUC=0.6310
    Decision Tree              Acc=0.5595  F1=0.43

In [ ]:
# ════════════════════════════════════════════════
# CELL 7 — Compute Deltas: SLTE vs No-SLTE
# ════════════════════════════════════════════════
df_no   = pd.read_csv('tstr_results.csv')
df_sl   = pd.read_csv('slte_results.csv')
TREE    = ['Decision Tree','Random Forest','Gradient Boosting']
PARAM   = ['Logistic Regression','Naive Bayes']

print('='*65)
print('DELTA TABLE — SLTE vs No-SLTE (AUC-ROC improvement)')
print('='*65)

delta_rows = []
for clf in TREE + PARAM:
    tag = 'TREE ' if clf in TREE else 'PARAM'
    for ratio in MIXING_RATIOS:
        no_val  = df_no[(df_no['classifier']==clf)&(df_no['synthetic_ratio']==ratio)]['auc_roc'].values[0]
        sl_val  = df_sl[(df_sl['classifier']==clf)&(df_sl['synthetic_ratio']==ratio)]['auc_roc'].values[0]
        delta   = round(sl_val - no_val, 4)
        delta_rows.append({'classifier':clf,'type':tag,'synthetic_ratio':ratio,
                           'auc_no_slte':no_val,'auc_slte':sl_val,'delta_auc':delta})

delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv('delta_slte_vs_noSlte.csv', index=False)

# Print summary at 100% synthetic (most important comparison)
print(f'\n  At 100% Synthetic (most critical mixing ratio):')
print(f'  {"Classifier":<25} {"No-SLTE AUC":>12} {"SLTE AUC":>10} {"Delta":>8}')
print(f'  {"-"*60}')
for clf in TREE + PARAM:
    row = delta_df[(delta_df['classifier']==clf)&(delta_df['synthetic_ratio']==1.0)].iloc[0]
    sign = '+' if row['delta_auc'] >= 0 else ''
    print(f'  {clf:<25} {row["auc_no_slte"]:>12.4f} {row["auc_slte"]:>10.4f} {sign}{row["delta_auc"]:>7.4f}')

# Overall improvement stats
tree_delta  = delta_df[(delta_df['type']=='TREE ')&(delta_df['synthetic_ratio']==1.0)]['delta_auc'].mean()
param_delta = delta_df[(delta_df['type']=='PARAM')&(delta_df['synthetic_ratio']==1.0)]['delta_auc'].mean()
print(f'\n  Avg SLTE gain (tree-based):  {tree_delta:+.4f} AUC')
print(f'  Avg SLTE gain (parametric):  {param_delta:+.4f} AUC')


summary = {
    'run_timestamp'         : datetime.now().isoformat(),
    'lcs_mean'              : round(float(lcs.mean()), 4),
    'lcs_std'               : round(float(lcs.std()),  4),
    'low_confidence_count'  : int(low_mask.sum()),
    'mid_confidence_count'  : int(mid_mask.sum()),
    'high_confidence_count' : int(high_mask.sum()),
    'labels_corrected'      : int(n_corrected),
    'correction_rate_pct'   : round(n_corrected/len(y_synth)*100, 2),
    'avg_slte_gain_tree_auc': round(float(tree_delta),  4),
    'avg_slte_gain_param_auc':round(float(param_delta), 4),
}
with open('day2_summary_log.json','w') as f:
    json.dump(summary, f, indent=2)


DELTA TABLE — SLTE vs No-SLTE (AUC-ROC improvement)

  At 100% Synthetic (most critical mixing ratio):
  Classifier                 No-SLTE AUC   SLTE AUC    Delta
  ------------------------------------------------------------
  Decision Tree                   0.6467     0.6742 + 0.0275
  Random Forest                   0.6399     0.6873 + 0.0474
  Gradient Boosting               0.6843     0.7146 + 0.0303
  Logistic Regression             0.5964     0.5988 + 0.0024
  Naive Bayes                     0.5803     0.6119 + 0.0316

  Avg SLTE gain (tree-based):  +0.0351 AUC
  Avg SLTE gain (parametric):  +0.0170 AUC


In [ ]:
# ════════════════════════════════════════════════
# CELL 8 — Download all Day 2 files at once
# ════════════════════════════════════════════════
import shutil
shutil.make_archive('day2_all_outputs', 'zip', '.', '.')
from google.colab import files
files.download('day2_all_outputs.zip')
print('✅ ZIP downloaded — extract and keep all .csv and .json files for Day 3')

RuntimeError: File size too large, try using force_zip64

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

# Step 1: Take your real training data
# X_train_real, y_train_real (already have this from Cell 2)

# Step 2: Create RANDOMLY mislabeled synthetic labels
# (simulate random noise — NOT CTGAN, just random flips)
np.random.seed(42)
n_synthetic = len(X_synth)  # <-- FIXED: changed from synthetic_data
random_labels = np.random.choice([0, 1], size=n_synthetic,
                                 p=[0.8, 0.2])  # same base rate as real data

# Step 3: Apply YOUR EXACT same kNN correction
# to these randomly mislabeled samples
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_real, y_train_real)  # <-- FIXED: removed _scaled if not scaled earlier

# Find which randomly labeled samples would be "corrected"
knn_preds = knn.predict(X_synth)  # <-- FIXED: changed from X_synth_scaled
random_corrections = random_labels != knn_preds

# Step 4: Count directional flips in RANDOM baseline
random_1to0 = np.sum(
    (random_labels[random_corrections] == 1) &
    (knn_preds[random_corrections] == 0)
)
random_0to1 = np.sum(
    (random_labels[random_corrections] == 0) &
    (knn_preds[random_corrections] == 1)
)
total_random_corrections = random_corrections.sum()
baseline_1to0_pct = 100 * random_1to0 / total_random_corrections

print(f"BASELINE (random_labels): {baseline_1to0_pct:.1f}% are 1→0 flips")
print(f"YOUR CTGAN result:        96.6% are 1→0 flips")
print(f"Difference: {96.6 - baseline_1to0_pct:.1f}%")

BASELINE (random_labels): 71.1% are 1→0 flips
YOUR CTGAN result:        96.6% are 1→0 flips
Difference: 25.5%
